# 04 · Lire une option chain avec pandas

> 📘 Notebook compagnon du cours [Les grecques en finance](https://pgu.github.io/pi-gamma-upsilon/). Le code Python reproduit ce que font les playgrounds du site, avec les **mêmes conventions de desk** : vega par point de vol, theta par jour, rho par point de taux.
>
> **Installation** (une fois) : `pip install -r requirements.txt`, puis `jupyter lab`. Lance ce notebook depuis le dossier `notebooks/`.

**Leçon liée :** « Marché : lire une option chain ».

**Objectifs :** construire une chaîne d'options **simulée** (smile + bid/ask), en tirer le *mouvement implicite*, le *25-delta put* et le skew, puis écrire une fonction qui calcule l'IV de chaque ligne : celle-ci fonctionnera telle quelle sur de **vraies** cotations (section 4).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import greeks as g          # notre petite bibliothèque (voir greeks.py)

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.4f}".format

## 1. Une chaîne simulée

Spot 100, 30 jours, IV ATM 20 %, skew 0,4 : mêmes paramètres que dans le cours. Le spread bid/ask est de l'ordre de 5 % du prix (minimum 0,05) et les prix sont arrondis au tick de 0,05.

In [2]:
S0, T, r = 100, 30 / 365, 0.03
tick = lambda x: np.round(x / 0.05) * 0.05

def build_chain(S0=S0, T=T, r=r, atm=0.20, skew=0.4, curv=1.0, strikes=range(80, 121, 5)):
    rows = []
    for K in strikes:
        iv = float(g.smile_iv(S0, K, T, atm, skew, curv))
        row = {"strike": K, "iv": iv}
        for kind in ("call", "put"):
            o = {k: float(v) for k, v in g.greeks(S0, K, T, r, iv, kind).items()}
            spread = max(0.05, tick(0.05 * o["price"]))
            row |= {f"{kind}_bid": max(0.0, tick(o["price"] - spread / 2)),
                    f"{kind}_ask": max(0.05, tick(o["price"] + spread / 2)),
                    f"{kind}_mid": o["price"], f"{kind}_delta": o["delta"]}
        rows.append(row)
    return pd.DataFrame(rows).set_index("strike")

chain = build_chain()
chain[["call_bid", "call_ask", "call_delta", "iv", "put_bid", "put_ask", "put_delta"]]

,call_bid,call_ask,call_delta,iv,put_bid,put_ask,put_delta
strike,,,,,,,
80,19.7000,20.7000,0.9930,0.3264,0.0000,0.0500,-0.0070
85,14.9000,15.6500,0.9812,0.2822,0.0500,0.1000,-0.0188
90,10.1500,10.6500,0.9402,0.2473,0.1500,0.2000,-0.0598
95,5.7500,6.0500,0.8114,0.2203,0.6500,0.7000,-0.1886
100,2.3500,2.4500,0.5286,0.2000,2.1000,2.2000,-0.4714
105,0.5500,0.6000,0.1994,0.1856,5.2000,5.4500,-0.8006
110,0.0500,0.1000,0.0351,0.1763,9.5500,10.0500,-0.9649
115,0.0000,0.0500,0.0028,0.1715,14.3500,15.1000,-0.9972
120,0.0000,0.0500,0.0001,0.1706,19.2000,20.2000,-0.9999


## 2. Les quatre exercices de la leçon

**① Mouvement implicite** : prix du straddle ATM / spot.

In [3]:
atm = chain.loc[100]
straddle = atm["call_mid"] + atm["put_mid"]
print(f"Straddle ATM (milieu) = {straddle:.2f}  →  mouvement implicite ±{straddle / S0:.1%}")
print(f"Contrôle 0,8 × σ × √T × S = {0.8 * atm['iv'] * np.sqrt(T) * S0:.2f}")

Straddle ATM (milieu) = 4.57  →  mouvement implicite ±4.6%
Contrôle 0,8 × σ × √T × S = 4.59


**② 25-delta put** : le strike dont le delta de put est le plus proche de −0,25.

In [4]:
k25 = (chain["put_delta"] + 0.25).abs().idxmin()
print(f"25-delta put ≈ strike {k25} (delta {chain.loc[k25, 'put_delta']:.2f}), OTM de {1 - k25 / S0:.0%}")

25-delta put ≈ strike 95 (delta -0.19), OTM de 5%


**③ Skew** : IV du put 90 − IV du call 110.

In [5]:
print(f"Skew = {(chain.loc[90, 'iv'] - chain.loc[110, 'iv']) * 100:.1f} points de vol")

Skew = 7.1 points de vol


**④ Coût d'entrée/sortie** du straddle ATM : acheter à l'ask, revendre au bid.

In [6]:
cost = (atm["call_ask"] + atm["put_ask"]) - (atm["call_bid"] + atm["put_bid"])
print(f"Aller-retour = {cost:.2f}  ({cost / straddle:.1%} de la valeur du straddle)")

Aller-retour = 0.20  (4.4% de la valeur du straddle)


## 3. De la cotation à l'IV : une fonction réutilisable

Sur un écran de marché on voit des **prix** (bid/ask). Cette fonction ajoute une colonne d'IV calculée sur le prix milieu (en ignorant les lignes sans bid). On la teste sur la chaîne simulée, en utilisant les options **OTM** comme le font les desks (puts sous le spot, calls au-dessus) : on doit retrouver l'IV qui a servi à fabriquer la chaîne.

In [7]:
def add_iv(df, S, T, r, kind, bid="bid", ask="ask"):
    """df : index = strike, colonnes bid/ask. kind : 'call'/'put' ou une série de types (un par ligne).
    Ajoute 'mid' et 'iv_calc' (NaN si pas de bid ou prix hors bornes)."""
    out = df.copy()
    out["mid"] = (out[bid] + out[ask]) / 2
    kinds = [kind] * len(out) if isinstance(kind, str) else list(kind)
    out["iv_calc"] = [g.implied_vol(m, S, K, T, r, k) if b > 0 else np.nan
                      for m, K, k, b in zip(out["mid"], out.index, kinds, out[bid])]
    return out

otm_kind = np.where(chain.index < S0, "put", "call")
otm = pd.DataFrame({
    "type": otm_kind,
    "bid": np.where(otm_kind == "put", chain["put_bid"], chain["call_bid"]),
    "ask": np.where(otm_kind == "put", chain["put_ask"], chain["call_ask"]),
}, index=chain.index)
otm = add_iv(otm, S0, T, r, otm["type"])
otm["iv_modèle"] = chain["iv"]
otm["écart (pts)"] = (otm["iv_calc"] - otm["iv_modèle"]) * 100
otm

,type,bid,ask,mid,iv_calc,iv_modèle,écart (pts)
strike,,,,,,,
80,put,0.0000,0.0500,0.0250,NaN,0.3264,NaN
85,put,0.0500,0.1000,0.0750,0.2948,0.2822,1.2555
90,put,0.1500,0.2000,0.1750,0.2435,0.2473,-0.3766
95,put,0.6500,0.7000,0.6750,0.2204,0.2203,0.0113
100,call,2.3500,2.4500,2.4000,0.1992,0.2000,-0.0840
105,call,0.5500,0.6000,0.5750,0.1854,0.1856,-0.0256
110,call,0.0500,0.1000,0.0750,0.1790,0.1763,0.2675
115,call,0.0000,0.0500,0.0250,NaN,0.1715,NaN
120,call,0.0000,0.0500,0.0250,NaN,0.1706,NaN


Les IV retrouvées collent au modèle pour les options liquides (autour de la monnaie). Les écarts apparaissent sur les options **très peu chères** : quand un prix vaut 0,05–0,10, un seul tick (0,05) change l'IV de plusieurs points, et sans bid (`NaN`) on ne peut rien calculer. C'est une vraie difficulté de la donnée de marché.

## 4. Avec de vraies données (optionnel)

Pour refaire ces exercices sur une vraie cotation différée, la bibliothèque `yfinance` (non officielle, à installer séparément : `pip install yfinance`) permet de télécharger une chaîne. Esquisse **à adapter et à vérifier** (l'API peut changer, et les données sont différées / parfois incomplètes) :

```python
import yfinance as yf

tk = yf.Ticker("SPY")                       # exemple de ticker
expiry = tk.options[1]                      # une échéance
chain_real = tk.option_chain(expiry)
calls_real = chain_real.calls[["strike", "bid", "ask", "impliedVolatility"]]

spot = tk.history(period="1d")["Close"].iloc[-1]
T_real = (pd.Timestamp(expiry) - pd.Timestamp.today()).days / 365
calls_real = calls_real[(calls_real.bid > 0) & (calls_real.ask > 0)]
calls_real = add_iv(calls_real.set_index("strike"), spot, T_real, r=0.04, kind="call")    # r : voir FRED
calls_real.assign(écart=lambda d: d.iv_calc - d.impliedVolatility)
```

Compare `iv_calc` avec la colonne `impliedVolatility` fournie : les écarts viennent du taux, des dividendes, du prix utilisé (milieu vs dernier prix) et de l'exercice anticipé (options américaines, voir notebook 06).

### À toi de jouer

1. Passe `skew` de 0,4 à 0 dans `build_chain` : que devient l'exercice ③ ? Et l'exercice ② (le 25-delta put se rapproche-t-il du 25-delta call) ?
2. Calcule le **risk reversal 25Δ** : IV du call 25Δ − IV du put 25Δ.
3. Allonge la maturité à 180 jours : le mouvement implicite grandit-il comme √T ?